# Lab 33: The Missing Ear

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 4, *The Language Engine*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-33.ipynb)

**What you will do:** have a friend snap their fingers from several positions around you, measure how many degrees off you point with both ears open versus one ear blocked, compare your two-ear accuracy with the 5-10 degrees the book reports, and run a numpy simulation of the same timing-difference trick to see why a single "ear" cannot localise sound at all.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

Use a proper earplug or your cupped palm to block one ear — never insert a finger or any other
object into the ear canal itself. If you try the headphone or binaural versions, keep the
volume moderate throughout.

## The experiment in the book

In Lab 33 you closed your eyes and pointed towards a finger-snap, first with both ears open
and then with one ear blocked. With both ears, most people localise sound to within about
5-10 degrees. Blocking one ear removes the timing and loudness differences between the ears,
and accuracy collapses — especially for sounds on the blocked side.

## Record your results

Have a friend stand at several known positions around you (for example, straight ahead, 45
degrees left, 90 degrees left, 45 degrees right) and snap their fingers while your eyes are
closed. Call out where you think the sound came from, and have your friend record the actual
and your estimated angle, for both ears open and one ear blocked.

In [ ]:
# example data: replace with your own trials, one row per snap
trials = pd.DataFrame({
    "condition": ["two ears", "two ears", "two ears", "two ears",
                  "one ear blocked", "one ear blocked", "one ear blocked", "one ear blocked"],
    "true_angle_deg": [0, 45, -45, 90, 0, 45, -45, 90],
    "estimated_angle_deg": [3, 50, -38, 85, 20, 60, 10, 40],
})
trials["abs_error_deg"] = (trials.estimated_angle_deg - trials.true_angle_deg).abs()
display(trials)

trials.groupby("condition").abs_error_deg.mean().plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3))
plt.ylabel("mean absolute error (degrees)"); plt.xticks(rotation=0); plt.show()

## Compare

Lab 33 reports that two-ear localisation is typically accurate to within 5-10 degrees.
Compare your own two-ear mean error with that range, and see how much worse the one-ear
condition is by comparison.

In [ ]:
two_ear = trials.loc[trials.condition == "two ears", "abs_error_deg"].mean()
one_ear = trials.loc[trials.condition == "one ear blocked", "abs_error_deg"].mean()
print(f"Your two-ear mean error: {two_ear:.0f} degrees")
print(f"Your one-ear mean error: {one_ear:.0f} degrees")
if two_ear <= 10:
    print("Inside the typical 5-10 degree range Lab 33 reports for two-ear localisation.")
else:
    print("Wider than the typical range -- a small number of trials, an echoey room, or"
          " front-back confusions (which the book notes are hard even with two ears) can"
          " all inflate this figure.")
print(f"One-ear error was {one_ear - two_ear:.0f} degrees larger, consistent with the"
      " collapse the book describes once the timing and loudness cues from the second ear"
      " are removed.")

## The AI side

The brainstem's interaural time difference (ITD) circuits work like a coincidence detector,
comparing the tiny timing gap between the two ears. Below, a numpy simulation builds a toy
version of the same trick: a short click arrives at two simulated "microphones" a human
head-width apart, and a cross-correlation function estimates the direction from the timing
gap alone — the computational principle behind microphone-array beamforming (Van Veen &
Buckley, 1988). Then the right-hand microphone is replaced with pure noise, simulating a
blocked ear, to see what happens to the estimate.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
speed_of_sound = 343.0   # m/s, speed of sound in air
mic_separation = 0.18    # m, roughly the distance between human ears
sr = 96000                # Hz, a high simulated sample rate for sub-sample timing precision

def simulate_signals(true_angle_deg, snr_db=15, both_ears=True, seed=0):
    t = np.arange(int(sr * 0.01)) / sr
    pulse = np.sin(2 * np.pi * 1000 * t) * np.exp(-t / 0.002)  # a short, sharp click
    itd = mic_separation * np.sin(np.radians(true_angle_deg)) / speed_of_sound
    shift_samples = itd * sr
    idx = np.arange(len(pulse))
    delayed = np.interp(idx - shift_samples, idx, pulse, left=0.0, right=0.0)

    rng = np.random.default_rng(seed)
    noise_power = np.mean(pulse ** 2) / (10 ** (snr_db / 10))
    left = pulse + rng.normal(0, np.sqrt(noise_power), len(pulse))
    clean_right = delayed if both_ears else np.zeros_like(pulse)
    right = clean_right + rng.normal(0, np.sqrt(noise_power), len(pulse))
    return left, right

def estimate_angle(left, right):
    corr = np.correlate(right, left, mode="full")
    lags = np.arange(-len(left) + 1, len(left))
    estimated_itd = lags[np.argmax(corr)] / sr
    ratio = np.clip(estimated_itd * speed_of_sound / mic_separation, -1, 1)
    return np.degrees(np.arcsin(ratio))

rows = []
for angle in [-60, -30, 0, 30, 60]:
    two_ear_est = estimate_angle(*simulate_signals(angle, both_ears=True, seed=angle + 100))
    one_ear_est = estimate_angle(*simulate_signals(angle, both_ears=False, seed=angle + 200))
    rows.append({"true_angle_deg": angle, "two_ear_estimate": round(two_ear_est, 1),
                 "one_ear_estimate (no real ITD to find)": round(one_ear_est, 1)})

sim = pd.DataFrame(rows)
display(sim)
error = (sim.two_ear_estimate - sim.true_angle_deg).abs().mean()
print(f"Mean two-ear localisation error in the simulation: {error:.1f} degrees")

The two-ear column should track the true angle fairly closely; the one-ear column is close
to arbitrary, because with only one working channel there is no timing gap left to measure —
the "estimate" is noise dressed up as a number, just as your own one-ear pointing became
unreliable. The simulation is a deliberately simplified analogue: it uses one clean click at
one fixed loudness cue (interaural level differences, which matter for high-frequency
sounds, are not modelled at all), it ignores head shadow and pinna cues, and cross-correlation
on a single click is far cruder than the population of coincidence-detector neurons in the
real medial superior olive.

## Questions to think about

1. The simulation's estimate_angle function only uses timing (an ITD-based method). Real sound localisation also uses interaural level differences (ILDs), which dominate for high-frequency sounds. Based on the book's account of why ILDs matter more at high frequencies, what would you expect to happen to this simulation's accuracy if the click were a very high-frequency chirp instead of the 1000 Hz tone used here?
2. Front-back confusions are hard even with two ears, because the interaural cues are nearly identical for a sound directly ahead and directly behind. Does the simulation above, as written, have any way to tell front from back? Why or why not?
3. In your own recorded trials, was the collapse in accuracy with one ear blocked worse for sounds on the blocked side than for sounds on the open side? What does the book's account of coincidence detection in the medial superior olive predict about this asymmetry?
4. Beamforming microphone arrays in smart speakers use more than two microphones. Based on how the simulation estimates direction from a single pair, what advantage might a third microphone add?

## Challenge

Test front-back confusion specifically: have your friend snap directly in front of you and
then directly behind you, at the same distance, with both ears open. Log how often you
correctly tell front from back compared with how often you correctly tell left from right at
the same angle offset.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-33.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")